In [15]:
import pandas as pd

# 读取订单明细表 (包含 price, freight_value, product_id)
items = pd.read_csv('olist_order_items_dataset.csv')

# 读取产品表 (包含 product_id, product_category_name)
products = pd.read_csv('olist_products_dataset.csv')

In [16]:
# merge 的意思是合并，on='product_id' 说明根据 ID 匹配
# how='left' 表示以左边的 items 表为准，保证订单不丢失
merged_df = pd.merge(items, products, on='product_id', how='left')

# 运行这一行看看结果，你会发现每行后面都多了产品分类列
merged_df.head()
# 筛选出属于‘garden_tools’（园林工具）的数据
garden_data = merged_df[merged_df['product_category_name'] == 'ferramentas_jardim']
# 算一下这部分产品的总销售额
total_sales = garden_data['price'].sum()


print(f"园林工具类目的总销售额是: {total_sales}")

园林工具类目的总销售额是: 485256.45999999996


In [17]:
garden_data = merged_df[merged_df['product_category_name'] == 'ferramentas_jardim'].copy() #复制脱离原数据的新数据
# 计算运费占售价的比例
garden_data['freight_ratio'] = garden_data['freight_value'] / garden_data['price']
#新加一列，计算运费比例
# 看看平均运费占比
print(garden_data['freight_ratio'].mean())

0.33202687790160623


In [20]:
# 1. 读取客户表
customers = pd.read_csv('olist_customers_dataset.csv')

# 2. 将之前的 garden_data 与 customers 合并，获取地理位置
# 注意：我们要用 .copy() 保证它是独立的数据帧
# 第一步：读入订单表（它是中转站）
orders = pd.read_csv('olist_orders_dataset.csv')

# 第二步：先把 garden_data 和 orders 合并，拿到 customer_id
# 这样你的 garden_data 里就有了客户 ID
garden_with_order = pd.merge(garden_data, orders[['order_id', 'customer_id']], on='order_id', how='left')

# 第三步：现在可以用 customer_id 去连客户表了
garden_geo = pd.merge(garden_with_order, customers, on='customer_id', how='left').copy()

# 验证一下
print(garden_geo.columns) # 看看现在是不是同时有了 freight_ratio 和 customer_state

# 3. 按州分组，计算运费占比的平均值，并按从高到低排序
state_freight = garden_geo.groupby('customer_state')['freight_ratio'].mean().sort_values(ascending=False)

print("各州平均运费占比（从高到低）：")
print(state_freight)
state_freight.head()


Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value',
       'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'freight_ratio', 'customer_id', 'customer_unique_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state'],
      dtype='object')
各州平均运费占比（从高到低）：
customer_state
MA    0.847828
RR    0.677535
RO    0.642345
AP    0.631404
SE    0.609715
PB    0.591745
RN    0.579262
TO    0.576287
PA    0.520855
AM    0.490833
PE    0.470003
CE    0.416761
ES    0.409540
AL    0.404736
PI    0.403613
BA    0.400005
DF    0.386918
MS    0.374887
SC    0.370898
RJ    0.349464
PR    0.341464
RS    0.318782
MT    0.316651
MG    0.314931
GO    0.283513
SP    0.278607
AC    0.203167
Name: freight_ratio, dtype: float64


customer_state
MA    0.847828
RR    0.677535
RO    0.642345
AP    0.631404
SE    0.609715
Name: freight_ratio, dtype: float64